In [1]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

filename = list(uploaded.keys())[0]
print("업로드 파일:", filename)

xls = pd.ExcelFile(filename)

print("\n시트 목록:")
print(xls.sheet_names)

Saving 사회적+지원망_20260916171619.xlsx to 사회적+지원망_20260916171619.xlsx
업로드 파일: 사회적+지원망_20260916171619.xlsx

시트 목록:
['데이터', '메타정보']


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [2]:
for sheet in xls.sheet_names:
    df_temp = pd.read_excel(
        filename,
        sheet_name=sheet,
        header=None
    )

    print(f"\n===== {sheet} =====")
    print("크기:", df_temp.shape)

    display(df_temp.head(12))


===== 데이터 =====
크기: (58, 10)


,0,1,2,3,4,5,6,7,8,9
0,구분별(1),구분별(2),2025,2025,2025,2025,2025,2025,2025,2025
1,구분별(1),구분별(2),몸이 아플 때 보살펴 줄 수 있는 사람,몸이 아플 때 보살펴 줄 수 있는 사람,갑자기 금전적인 도움이 필요할 때 돈을 빌려 줄 사람,갑자기 금전적인 도움이 필요할 때 돈을 빌려 줄 사람,낙심하거나 우울해서 이야기할 사람,낙심하거나 우울해서 이야기할 사람,어린 자녀 또는 돌봄이 필요한 가족을 맡겨야 할 때 돌봐줄 사람,어린 자녀 또는 돌봄이 필요한 가족을 맡겨야 할 때 돌봐줄 사람
2,구분별(1),구분별(2),있다,없다,있다,없다,있다,없다,있다,없다
3,서울시,소계,95.9,4.1,84.3,15.7,94.7,5.3,78.6,21.4
4,성별,남성,95.5,4.5,84.4,15.6,94.4,5.6,78.9,21.1
5,NaN,여성,96.3,3.7,84.3,15.7,94.9,5.1,78.3,21.7
6,연령별,10대,98.9,1.1,90.1,9.9,96.7,3.3,64.9,35.1
7,NaN,20대,97.2,2.8,87.6,12.4,95.6,4.4,84.6,15.4
8,NaN,30대,96.1,3.9,89.1,10.9,96.3,3.7,87.4,12.6
9,NaN,40대,96.4,3.6,84.9,15.1,94.8,5.2,83.7,16.3



===== 메타정보 =====
크기: (15, 2)


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,0,1
0,○ 통계표ID,DT_201011_B100005
1,○ 통계표명,사회적 지원망
2,○ 조회기간,[년] 2025~2025
3,○ 출처,"서울특별시, 서울서베이"
4,○ 자료다운일자,2026.09.16 17:16
5,○ 통계표URL,https://stat.eseoul.go.kr/statHtml/statHtml.do...
6,NaN,* 시스템 개편 시 통계표 URL은 달라질 수 있음
7,○ 단위,%
8,○ 주석,NaN
9,통계표,* 조회되지 않는 연도는 해당연도 조사 제외


In [3]:
social_raw = pd.read_excel(
    filename,
    sheet_name="데이터",
    header=None
)

# 실제 데이터 영역
social_data = social_raw.iloc[3:].copy()

# 첫 번째 분류열의 병합셀 복원
social_data[0] = social_data[0].ffill()

print("분류 목록:")
print(social_data[0].unique())

print("\n강남 관련 행:")
display(
    social_data[
        social_data[1].astype(str).str.contains(
            "강남",
            na=False
        )
    ]
)

분류 목록:
['서울시' '성별' '연령별' '학력별' '소득별' '혼인상태별' '지역대분류' '지역소분류' '직업분류']

강남 관련 행:


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,0,1,2,3,4,5,6,7,8,9
51,지역소분류,강남구,98.6,1.4,91.8,8.2,96.3,3.7,51.7,48.3


In [4]:
district_rows = social_data[
    social_data[0].astype(str).str.contains(
        "자치구",
        na=False
    )
]

print("자치구 관련 행 수:", len(district_rows))

display(
    district_rows[[0, 1]]
)

자치구 관련 행 수: 0


,0,1


In [5]:
gangnam_social_support = social_data[
    social_data[1] == "강남구"
].copy()

gangnam_support_bg = pd.DataFrame([{
    "region": "강남구",
    "year": 2025,

    "care_support_present_pct": float(gangnam_social_support.iloc[0, 2]),
    "care_support_absent_pct": float(gangnam_social_support.iloc[0, 3]),

    "financial_support_present_pct": float(gangnam_social_support.iloc[0, 4]),
    "financial_support_absent_pct": float(gangnam_social_support.iloc[0, 5]),

    "emotional_support_present_pct": float(gangnam_social_support.iloc[0, 6]),
    "emotional_support_absent_pct": float(gangnam_social_support.iloc[0, 7]),

    "family_care_support_present_pct": float(gangnam_social_support.iloc[0, 8]),
    "family_care_support_absent_pct": float(gangnam_social_support.iloc[0, 9])
}])

display(gangnam_support_bg)

,region,year,care_support_present_pct,care_support_absent_pct,financial_support_present_pct,financial_support_absent_pct,emotional_support_present_pct,emotional_support_absent_pct,family_care_support_present_pct,family_care_support_absent_pct
0,강남구,2025,98.6,1.4,91.8,8.2,96.3,3.7,51.7,48.3


In [6]:
checks = {
    "care": (
        gangnam_support_bg["care_support_present_pct"].iloc[0]
        + gangnam_support_bg["care_support_absent_pct"].iloc[0]
    ),
    "financial": (
        gangnam_support_bg["financial_support_present_pct"].iloc[0]
        + gangnam_support_bg["financial_support_absent_pct"].iloc[0]
    ),
    "emotional": (
        gangnam_support_bg["emotional_support_present_pct"].iloc[0]
        + gangnam_support_bg["emotional_support_absent_pct"].iloc[0]
    ),
    "family_care": (
        gangnam_support_bg["family_care_support_present_pct"].iloc[0]
        + gangnam_support_bg["family_care_support_absent_pct"].iloc[0]
    )
}

print("있다 + 없다 검증:")
for key, value in checks.items():
    print(key, ":", value)

print("\n결측치:")
print(gangnam_support_bg.isna().sum())

있다 + 없다 검증:
care : 100.0
financial : 100.0
emotional : 100.0
family_care : 100.0

결측치:
region                             0
year                               0
care_support_present_pct           0
care_support_absent_pct            0
financial_support_present_pct      0
financial_support_absent_pct       0
emotional_support_present_pct      0
emotional_support_absent_pct       0
family_care_support_present_pct    0
family_care_support_absent_pct     0
dtype: int64


In [7]:
# 성별 + 연령별 행만 추출
gender_age_support = social_data[
    social_data[0].isin(["성별", "연령별"])
].copy()

print("shape:", gender_age_support.shape)

display(gender_age_support)

shape: (8, 10)


,0,1,2,3,4,5,6,7,8,9
4,성별,남성,95.5,4.5,84.4,15.6,94.4,5.6,78.9,21.1
5,성별,여성,96.3,3.7,84.3,15.7,94.9,5.1,78.3,21.7
6,연령별,10대,98.9,1.1,90.1,9.9,96.7,3.3,64.9,35.1
7,연령별,20대,97.2,2.8,87.6,12.4,95.6,4.4,84.6,15.4
8,연령별,30대,96.1,3.9,89.1,10.9,96.3,3.7,87.4,12.6
9,연령별,40대,96.4,3.6,84.9,15.1,94.8,5.2,83.7,16.3
10,연령별,50대,95.9,4.1,84.4,15.6,95.4,4.6,71.4,28.6
11,연령별,60세 이상,94.5,5.5,78.4,21.6,92.3,7.7,61.9,38.1


In [8]:
print("성별:")
print(
    gender_age_support[
        gender_age_support[0] == "성별"
    ][1].tolist()
)

print("\n연령별:")
print(
    gender_age_support[
        gender_age_support[0] == "연령별"
    ][1].tolist()
)

성별:
['남성', '여성']

연령별:
['10대', '20대', '30대', '40대', '50대', '60세 이상']


In [9]:
# 성별·연령별 Reference 정리
social_support_reference = gender_age_support.copy()

social_support_reference.columns = [
    "group_type",
    "group_name",
    "care_support_present_pct",
    "care_support_absent_pct",
    "financial_support_present_pct",
    "financial_support_absent_pct",
    "emotional_support_present_pct",
    "emotional_support_absent_pct",
    "family_care_support_present_pct",
    "family_care_support_absent_pct"
]

# 연도 추가
social_support_reference.insert(0, "year", 2025)

# 비율 컬럼 숫자형 변환
pct_cols = [
    "care_support_present_pct",
    "care_support_absent_pct",
    "financial_support_present_pct",
    "financial_support_absent_pct",
    "emotional_support_present_pct",
    "emotional_support_absent_pct",
    "family_care_support_present_pct",
    "family_care_support_absent_pct"
]

social_support_reference[pct_cols] = (
    social_support_reference[pct_cols]
    .apply(pd.to_numeric, errors="coerce")
)

display(social_support_reference)

,year,group_type,group_name,care_support_present_pct,care_support_absent_pct,financial_support_present_pct,financial_support_absent_pct,emotional_support_present_pct,emotional_support_absent_pct,family_care_support_present_pct,family_care_support_absent_pct
4,2025,성별,남성,95.5,4.5,84.4,15.6,94.4,5.6,78.9,21.1
5,2025,성별,여성,96.3,3.7,84.3,15.7,94.9,5.1,78.3,21.7
6,2025,연령별,10대,98.9,1.1,90.1,9.9,96.7,3.3,64.9,35.1
7,2025,연령별,20대,97.2,2.8,87.6,12.4,95.6,4.4,84.6,15.4
8,2025,연령별,30대,96.1,3.9,89.1,10.9,96.3,3.7,87.4,12.6
9,2025,연령별,40대,96.4,3.6,84.9,15.1,94.8,5.2,83.7,16.3
10,2025,연령별,50대,95.9,4.1,84.4,15.6,95.4,4.6,71.4,28.6
11,2025,연령별,60세 이상,94.5,5.5,78.4,21.6,92.3,7.7,61.9,38.1


In [10]:
check_pairs = {
    "care": (
        "care_support_present_pct",
        "care_support_absent_pct"
    ),
    "financial": (
        "financial_support_present_pct",
        "financial_support_absent_pct"
    ),
    "emotional": (
        "emotional_support_present_pct",
        "emotional_support_absent_pct"
    ),
    "family_care": (
        "family_care_support_present_pct",
        "family_care_support_absent_pct"
    )
}

for name, (present, absent) in check_pairs.items():
    social_support_reference[f"{name}_sum"] = (
        social_support_reference[present]
        + social_support_reference[absent]
    )

print("shape:", social_support_reference.shape)

print("\n비율 합계 검증:")
for name in check_pairs:
    col = f"{name}_sum"
    print(
        name,
        "min =", social_support_reference[col].min(),
        "max =", social_support_reference[col].max()
    )

print("\n결측치 수:")
print(social_support_reference[pct_cols].isna().sum().sum())

shape: (8, 15)

비율 합계 검증:
care min = 100.0 max = 100.0
financial min = 100.0 max = 100.0
emotional min = 100.0 max = 100.0
family_care min = 100.0 max = 100.0

결측치 수:
0


In [11]:
# 검증용 컬럼 제거
sum_cols = [
    "care_sum",
    "financial_sum",
    "emotional_sum",
    "family_care_sum"
]

social_support_reference_final = (
    social_support_reference
    .drop(columns=sum_cols)
    .copy()
)

# 저장
gangnam_support_bg.to_csv(
    "gangnam_social_support_background_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

social_support_reference_final.to_csv(
    "seoul_social_support_gender_age_reference_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "강남구 Background:",
    gangnam_support_bg.shape
)

print(
    "서울시 성별·연령별 Reference:",
    social_support_reference_final.shape
)

강남구 Background: (1, 10)
서울시 성별·연령별 Reference: (8, 11)


In [12]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

filename = list(uploadloaded.keys())[0] if False else list(uploaded.keys())[0]

print("업로드 파일:", filename)

xls = pd.ExcelFile(filename)

print("\n시트 목록:")
print(xls.sheet_names)

for sheet in xls.sheet_names:
    df_temp = pd.read_excel(
        filename,
        sheet_name=sheet,
        header=None
    )

    print(f"\n===== {sheet} =====")
    print("크기:", df_temp.shape)

    display(df_temp.head(15))

Saving 사회적+신뢰_20260916171330.xlsx to 사회적+신뢰_20260916171330.xlsx
업로드 파일: 사회적+신뢰_20260916171330.xlsx

시트 목록:
['데이터', '메타정보']

===== 데이터 =====
크기: (57, 9)


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,0,1,2,3,4,5,6,7,8
0,구분별(1),구분별(2),2025,2025,2025,2025,2025,2025,2025
1,구분별(1),구분별(2),종합,가족,이웃,친구,"공공 기관(서울시, 구청 등)",처음 만난 낯선 사람,국내 거주 외국인
2,서울시,소계,5.32,8.81,4.64,7.29,5.55,2.75,2.86
3,성별,남성,5.27,8.75,4.54,7.25,5.51,2.74,2.83
4,NaN,여성,5.36,8.86,4.73,7.32,5.59,2.76,2.89
5,연령별,10대,5.36,9.16,4.52,7.89,5.37,2.51,2.7
6,NaN,20대,5.27,8.9,4.35,7.57,5.45,2.57,2.77
7,NaN,30대,5.3,8.9,4.4,7.41,5.64,2.63,2.83
8,NaN,40대,5.36,8.84,4.62,7.25,5.64,2.84,2.98
9,NaN,50대,5.3,8.74,4.73,7.18,5.53,2.79,2.86



===== 메타정보 =====
크기: (11, 2)


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,0,1
0,○ 통계표ID,DT_201011_B100002
1,○ 통계표명,사회적 신뢰
2,○ 조회기간,[년] 2025~2025
3,○ 출처,"서울특별시, 서울서베이"
4,○ 자료다운일자,2026.09.16 17:13
5,○ 통계표URL,https://stat.eseoul.go.kr/statHtml/statHtml.do...
6,NaN,* 시스템 개편 시 통계표 URL은 달라질 수 있음
7,○ 단위,점
8,○ 주석,NaN
9,통계표,※BASE : 전체


In [13]:
trust_raw = pd.read_excel(
    filename,
    sheet_name="데이터",
    header=None
)

# 실제 데이터
trust_data = trust_raw.iloc[2:].copy()

# 병합된 첫 번째 분류열 복원
trust_data[0] = trust_data[0].ffill()

print("분류 목록:")
print(trust_data[0].unique())

print("\n강남구 행:")
display(
    trust_data[
        trust_data[1].astype(str).str.strip() == "강남구"
    ]
)

분류 목록:
['서울시' '성별' '연령별' '학력별' '소득별' '혼인상태별' '지역대분류' '지역소분류' '직업분류']

강남구 행:


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,0,1,2,3,4,5,6,7,8
50,지역소분류,강남구,5.13,9.43,4.22,7.57,5.41,1.95,2.18


In [14]:
# 강남구 행 추출
gangnam_trust = trust_data[
    trust_data[1].astype(str).str.strip() == "강남구"
].copy()

gangnam_trust_bg = pd.DataFrame([{
    "region": "강남구",
    "year": 2025,
    "trust_overall": float(gangnam_trust.iloc[0, 2]),
    "trust_family": float(gangnam_trust.iloc[0, 3]),
    "trust_neighbor": float(gangnam_trust.iloc[0, 4]),
    "trust_friend": float(gangnam_trust.iloc[0, 5]),
    "trust_public_institution": float(gangnam_trust.iloc[0, 6]),
    "trust_stranger": float(gangnam_trust.iloc[0, 7]),
    "trust_foreigner": float(gangnam_trust.iloc[0, 8])
}])

display(gangnam_trust_bg)

,region,year,trust_overall,trust_family,trust_neighbor,trust_friend,trust_public_institution,trust_stranger,trust_foreigner
0,강남구,2025,5.13,9.43,4.22,7.57,5.41,1.95,2.18


In [15]:
trust_cols = [
    "trust_overall",
    "trust_family",
    "trust_neighbor",
    "trust_friend",
    "trust_public_institution",
    "trust_stranger",
    "trust_foreigner"
]

print("shape:", gangnam_trust_bg.shape)

print("\n결측치 수:")
print(gangnam_trust_bg[trust_cols].isna().sum().sum())

print("\n0~10 범위 검증:")
for col in trust_cols:
    value = gangnam_trust_bg[col].iloc[0]
    print(
        col,
        "=", value,
        "| 정상:", 0 <= value <= 10
    )

shape: (1, 9)

결측치 수:
0

0~10 범위 검증:
trust_overall = 5.13 | 정상: True
trust_family = 9.43 | 정상: True
trust_neighbor = 4.22 | 정상: True
trust_friend = 7.57 | 정상: True
trust_public_institution = 5.41 | 정상: True
trust_stranger = 1.95 | 정상: True
trust_foreigner = 2.18 | 정상: True


In [16]:
# 성별 + 연령별 행 추출
gender_age_trust = trust_data[
    trust_data[0].isin(["성별", "연령별"])
].copy()

print("shape:", gender_age_trust.shape)

print("\n성별:")
print(
    gender_age_trust[
        gender_age_trust[0] == "성별"
    ][1].tolist()
)

print("\n연령별:")
print(
    gender_age_trust[
        gender_age_trust[0] == "연령별"
    ][1].tolist()
)

display(gender_age_trust)

shape: (8, 9)

성별:
['남성', '여성']

연령별:
['10대', '20대', '30대', '40대', '50대', '60세 이상']


,0,1,2,3,4,5,6,7,8
3,성별,남성,5.27,8.75,4.54,7.25,5.51,2.74,2.83
4,성별,여성,5.36,8.86,4.73,7.32,5.59,2.76,2.89
5,연령별,10대,5.36,9.16,4.52,7.89,5.37,2.51,2.7
6,연령별,20대,5.27,8.9,4.35,7.57,5.45,2.57,2.77
7,연령별,30대,5.3,8.9,4.4,7.41,5.64,2.63,2.83
8,연령별,40대,5.36,8.84,4.62,7.25,5.64,2.84,2.98
9,연령별,50대,5.3,8.74,4.73,7.18,5.53,2.79,2.86
10,연령별,60세 이상,5.32,8.66,4.89,7.05,5.56,2.88,2.9


In [17]:
social_trust_reference = gender_age_trust.copy()

social_trust_reference.columns = [
    "group_type",
    "group_name",
    "trust_overall",
    "trust_family",
    "trust_neighbor",
    "trust_friend",
    "trust_public_institution",
    "trust_stranger",
    "trust_foreigner"
]

social_trust_reference.insert(0, "year", 2025)

# 숫자형 변환
trust_ref_cols = [
    "trust_overall",
    "trust_family",
    "trust_neighbor",
    "trust_friend",
    "trust_public_institution",
    "trust_stranger",
    "trust_foreigner"
]

social_trust_reference[trust_ref_cols] = (
    social_trust_reference[trust_ref_cols]
    .apply(pd.to_numeric, errors="coerce")
)

print("shape:", social_trust_reference.shape)
print("결측치:", social_trust_reference[trust_ref_cols].isna().sum().sum())

print("\n0~10 범위 밖 값:")
print(
    ((social_trust_reference[trust_ref_cols] < 0) |
     (social_trust_reference[trust_ref_cols] > 10))
    .sum()
    .sum()
)

display(social_trust_reference)

shape: (8, 10)
결측치: 0

0~10 범위 밖 값:
0


,year,group_type,group_name,trust_overall,trust_family,trust_neighbor,trust_friend,trust_public_institution,trust_stranger,trust_foreigner
3,2025,성별,남성,5.27,8.75,4.54,7.25,5.51,2.74,2.83
4,2025,성별,여성,5.36,8.86,4.73,7.32,5.59,2.76,2.89
5,2025,연령별,10대,5.36,9.16,4.52,7.89,5.37,2.51,2.70
6,2025,연령별,20대,5.27,8.90,4.35,7.57,5.45,2.57,2.77
7,2025,연령별,30대,5.30,8.90,4.40,7.41,5.64,2.63,2.83
8,2025,연령별,40대,5.36,8.84,4.62,7.25,5.64,2.84,2.98
9,2025,연령별,50대,5.30,8.74,4.73,7.18,5.53,2.79,2.86
10,2025,연령별,60세 이상,5.32,8.66,4.89,7.05,5.56,2.88,2.90


In [18]:
gangnam_trust_bg.to_csv(
    "gangnam_social_trust_background_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

social_trust_reference.to_csv(
    "seoul_social_trust_gender_age_reference_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "강남구 Social Trust Background:",
    gangnam_trust_bg.shape
)

print(
    "서울시 Social Trust Reference:",
    social_trust_reference.shape
)

강남구 Social Trust Background: (1, 9)
서울시 Social Trust Reference: (8, 10)


In [19]:
from google.colab import files

download_files = [
    "gangnam_social_support_background_2025.csv",
    "seoul_social_support_gender_age_reference_2025.csv",
    "gangnam_social_trust_background_2025.csv",
    "seoul_social_trust_gender_age_reference_2025.csv"
]

for file in download_files:
    files.download(file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>